In [33]:
import cv2
import imutils
import easyocr
import numpy as np
import matplotlib.pyplot as plt

In [56]:
pip install ultralytics

  Using cached opencv_python-4.13.0.92-cp37-abi3-win_amd64.whl.metadata (20 kB)
  Using cached ultralytics_thop-2.0.19-py3-none-any.whl.metadata (14 kB)
   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   ---------------- ----------------------- 0.5/1.3 MB 3.4 MB/s eta 0:00:01
   ---------------- ----------------------- 0.5/1.3 MB 3.4 MB/s eta 0:00:01
   ---------------- ----------------------- 0.5/1.3 MB 3.4 MB/s eta 0:00:01
   ---------------- ----------------------- 0.5/1.3 MB 3.4 MB/s eta 0:00:01
   ---------------- ----------------------- 0.5/1.3 MB 3.4 MB/s eta 0:00:01
   ------------------------ --------------- 0.8/1.3 MB 435.8 kB/s eta 0:00:02
   ---------------------------------------- 

ERROR: Could not install packages due to an OSError: [WinError 5] Access is denied: 'C:\\Users\\vv\\AppData\\Local\\Programs\\Python\\Python310\\Lib\\site-packages\\cv2\\cv2.pyd'
Consider using the `--user` option or check the permissions.



In [1]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

print("YOLO Loaded Successfully")

YOLO Loaded Successfully


In [1]:
import cv2
import easyocr
import imutils
from ultralytics import YOLO
import numpy as np

# =========================
# LOAD MODEL
# =========================
model = YOLO(r"C:\Users\vv\OneDrive\Desktop\PYTHON\opencv\license_plate_detector.pt")

reader = easyocr.Reader(['en'], gpu=False)

cap = cv2.VideoCapture(
    r"C:\Users\vv\OneDrive\Desktop\PYTHON\opencv\License Plate Detection Test.mp4"
)

last_plate_texts = {}

# =========================
def clean_text(text):
    return ''.join(filter(str.isalnum, text.upper()))

def choose_best_text(results):
    if not results:
        return ""
    results = [clean_text(r) for r in results]
    results = [r for r in results if len(r) >= 5]
    if not results:
        return ""
    return max(set(results), key=results.count)

# =========================
while True:

    ret, frame = cap.read()
    if not ret:
        break

    frame = imutils.resize(frame, width=900)
    display = frame.copy()

    results = model(frame)

    for result in results:
        for box in result.boxes:

            x1, y1, x2, y2 = map(int, box.xyxy[0])
            conf = float(box.conf[0])

            if conf < 0.5:
                continue

            # =========================
            # SAFE CROP + PADDING FIX
            # =========================
            h, w = frame.shape[:2]

            pad = 5  # small padding improves OCR
            x1 = max(0, x1 - pad)
            y1 = max(0, y1 - pad)
            x2 = min(w, x2 + pad)
            y2 = min(h, y2 + pad)

            plate = frame[y1:y2, x1:x2]
            if plate.size == 0:
                continue

            cv2.rectangle(display, (x1, y1), (x2, y2), (0, 255, 0), 2)

            # =========================
            # BETTER PREPROCESSING (NO CLAHE)
            # =========================

            gray = cv2.cvtColor(plate, cv2.COLOR_BGR2GRAY)

            # 1. light denoise
            gray = cv2.fastNlMeansDenoising(gray, None, 10, 7, 21)

            # 2. upscale using BEST interpolation
            gray = cv2.resize(
                gray,
                None,
                fx=5,
                fy=5,
                interpolation=cv2.INTER_LANCZOS4
            )

            # 3. gentle sharpening (safe)
            kernel = np.array([[0, -1, 0],
                               [-1, 5, -1],
                               [0, -1, 0]])
            gray = cv2.filter2D(gray, -1, kernel)

            # =========================
            # OCR
            # =========================
            ocr_results = reader.readtext(
                gray,
                detail=0,
                paragraph=False,
                contrast_ths=0.05,
                adjust_contrast=0.3
            )

            cleaned = [clean_text(t) for t in ocr_results]
            cleaned = [c for c in cleaned if len(c) >= 5]

            best_text = choose_best_text(cleaned)

            box_id = f"{x1//10}_{y1//10}"

            if best_text:
                if box_id not in last_plate_texts:
                    last_plate_texts[box_id] = best_text
                else:
                    if len(best_text) >= len(last_plate_texts[box_id]):
                        last_plate_texts[box_id] = best_text

            final_text = last_plate_texts.get(box_id, "")

            if final_text:
                print("Detected Plate:", final_text)

            cv2.imshow("Plate", gray)

    cv2.imshow("YOLO License Plate Detection", display)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

Using CPU. Note: This module is much faster with a GPU.



0: 384x640 2 license_plates, 422.9ms
Speed: 20.0ms preprocess, 422.9ms inference, 18.1ms postprocess per image at shape (1, 3, 384, 640)


C:\Users\vv\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Detected Plate: HAI3MRU

0: 384x640 2 license_plates, 286.4ms
Speed: 10.9ms preprocess, 286.4ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)
Detected Plate: HAI3MRU

0: 384x640 2 license_plates, 222.0ms
Speed: 6.8ms preprocess, 222.0ms inference, 2.6ms postprocess per image at shape (1, 3, 384, 640)
Detected Plate: HAIZMRU

0: 384x640 2 license_plates, 248.0ms
Speed: 6.3ms preprocess, 248.0ms inference, 2.0ms postprocess per image at shape (1, 3, 384, 640)
Detected Plate: HAI3MRU

0: 384x640 2 license_plates, 222.0ms
Speed: 7.4ms preprocess, 222.0ms inference, 1.9ms postprocess per image at shape (1, 3, 384, 640)
Detected Plate: HAIJMRU

0: 384x640 2 license_plates, 209.0ms
Speed: 8.0ms preprocess, 209.0ms inference, 1.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 license_plates, 189.8ms
Speed: 9.3ms preprocess, 189.8ms inference, 2.0ms postprocess per image at shape (1, 3, 384, 640)
Detected Plate: HAIJMRU

0: 384x640 2 license_plates, 326.6ms
S